# LLM & RAG Workflow

### Required libraries

In [18]:
!pip install langchain
!pip install langchain-community
!pip install sentence-transformers
!pip install -U langchain-huggingface jupyter
!pip install lmstudio
!pip install -U faiss-cpu
!pip install huggingface_hub[hf_xet]
# !pip install transformers torch accelerate bitsandbytes

In [19]:
from langchain.document_loaders import CSVLoader
from langchain.schema import Document
from langchain.text_splitter import CharacterTextSplitter
from langchain.document_loaders import AsyncChromiumLoader
from langchain.document_transformers import Html2TextTransformer
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

## Load data

- CSV to text format 

In [20]:
# Load the CSV file
DATASET_CSV_PATH = "../data/fighters_cleaned.csv"

# Load documents from the CSV
docs = CSVLoader(file_path=DATASET_CSV_PATH).load()

# Save the documents in text format
with open("../data/fighters.txt", "w") as f:
    for doc in docs:     
        f.write(doc.page_content + "\n")


In [21]:
with open("../data/fighters.txt", "r") as f:
    text = f.read()

In [22]:
text

"name: saul alvarez\nwins: 54\nlooses: 1\ndraws: 2\nko_rate: 0.632\nstance: Orthodox\nage: 32\ncountry: Mexico\nheight_cm: 174.95520000000002\nreach_cm: 178.9938\nname: rene alvarado\nwins: 32\nlooses: 8\ndraws: 0\nko_rate: 0.525\nstance: Orthodox\nage: 33\ncountry: Nicaragua\nheight_cm: 170.07840000000002\nreach_cm: 183.007\nname: daniel alicea\nwins: 30\nlooses: 7\ndraws: 2\nko_rate: 0.564\nstance: Orthodox\nage: 49\ncountry: Puerto Rico\nheight_cm: 173.1264\nreach_cm: 178.0032\nname: sadam ali\nwins: 27\nlooses: 3\ndraws: 0\nko_rate: 0.467\nstance: Orthodox\nage: 33\ncountry: United States\nheight_cm: 174.95520000000002\nreach_cm: 184.9882\nname: muhammad ali\nwins: 56\nlooses: 5\ndraws: 0\nko_rate: 0.607\nstance: Orthodox\nage: 80\ncountry: United States\nheight_cm: 191.1096\nreach_cm: 197.99300000000002\nname: juan zurita\nwins: 130\nlooses: 23\ndraws: 1\nko_rate: 0.299\nstance: Orthodox\nage: 105\ncountry: Mexico\nheight_cm: 164.8968\nreach_cm: 167.9956\nname: fritzie zivic\nwins

## RAG : Data Processing

In [23]:
# Split the text into chunks for better RAG performance
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separator="\n"
)

In [24]:
# Convert text chunks into Document objects
text_chunks = text_splitter.split_text(text)
documents = [Document(page_content=chunk) for chunk in text_chunks]

In [25]:
# Create embeddings and vector database
embedding_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2')
vectorized_db = FAISS.from_documents(
                            documents, embedding_model
                )

In [26]:
# Sauvegarder localement la base de données vectorisée
vectorized_db.save_local("faiss_vectorized_db")

## LLM Model

In [27]:
# Model setup with lightweight alternatives and proper error handling
def setup_llm_model():
    """Setup LLM model with multiple fallback options"""
    
    try:
        # Option 1: Try lightweight models first
        print("🔄 Trying lightweight models...")
        
        # Use a smaller, CPU-friendly model
        model_name = "distilgpt2"  # Very lightweight model
        
        # Create pipeline without device_map (CPU-friendly)
        hf_pipeline = pipeline(
            "text-generation",
            model=model_name,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            pad_token_id=50256,  # Set pad token for GPT-2
            device=-1  # Force CPU usage
        )
        
        # Wrap in LangChain
        model = HuggingFacePipeline(pipeline=hf_pipeline)
        print(f"✅ Lightweight model '{model_name}' loaded successfully!")
        return model, "lightweight"
        
    except Exception as e1:
        print(f"❌ Lightweight model failed: {e1}")
        
        try:
            # Option 2: Try even simpler approach
            print("🔄 Trying minimal setup...")
            
            # Use pipeline directly without explicit model loading
            hf_pipeline = pipeline(
                "text-generation",
                model="gpt2",  # Basic GPT-2
                max_length=200,
                temperature=0.8,
                device=-1
            )
            
            model = HuggingFacePipeline(pipeline=hf_pipeline)
            print("✅ Basic GPT-2 model loaded successfully!")
            return model, "basic"
            
        except Exception as e2:
            print(f"❌ Basic model failed: {e2}")
            
            # Option 3: Create a simple mock model for testing
            print("🔄 Creating mock model for testing...")
            
            class MockLLM:
                """Simple mock model for testing RAG system"""
                
                def _call(self, prompt: str, **kwargs) -> str:
                    return f"Mock response for: {prompt[:100]}..."
                
                def invoke(self, prompt: str, **kwargs) -> str:
                    return self._call(prompt, **kwargs)
                
                def __call__(self, prompt: str, **kwargs) -> str:
                    return self._call(prompt, **kwargs)
            
            model = MockLLM()
            print("✅ Mock model ready for testing!")
            return model, "mock"

In [28]:
# Setup the model
model, model_type = setup_llm_model()
print(f"Using model type: {model_type}")

🔄 Trying lightweight models...


Device set to use cpu


✅ Lightweight model 'distilgpt2' loaded successfully!
Using model type: lightweight


In [29]:
# Create retriever from the vector database
retriever = vectorized_db.as_retriever(
    search_type="similarity",
    search_kwargs={'k': 4}  # Retrieve top 4 most similar documents
)

# Create a custom prompt template for boxing Q&A
boxing_prompt_template = """
Vous êtes un expert en boxe. Utilisez les informations suivantes sur les boxeurs pour répondre à la question.
Si vous ne trouvez pas l'information dans le contexte fourni, dites-le clairement.

Contexte:
{context}

Question: {question}

Réponse détaillée:
"""

PROMPT = PromptTemplate(
    template=boxing_prompt_template,
    input_variables=["context", "question"]
)

# Create RAG chain with improved error handling
try:
    # First validate the model is working
    test_prompt = "Test prompt"
    if hasattr(model, '_call'):
        test_response = model._call(test_prompt)
    else:
        test_response = model.invoke(test_prompt)
    
    print(f"✅ Model validation successful: {test_response[:50]}...")
    
    # Now create the RAG chain
    qa_chain = RetrievalQA.from_chain_type(
        llm=model,
        chain_type="stuff",
        retriever=retriever,
        chain_type_kwargs={"prompt": PROMPT},
        return_source_documents=True
    )
    print("✅ RAG system ready for boxing questions!")
    
except Exception as e:
    print(f"❌ Error creating RAG chain: {e}")
    print("🔄 Using manual RAG implementation...")
    
    # Alternative: Manual RAG implementation
    def manual_rag(question: str):
        """Manual RAG implementation as fallback"""
        try:
            # Retrieve relevant documents
            docs = retriever.get_relevant_documents(question)
            
            # Combine context
            context = "\n".join([doc.page_content for doc in docs])
            
            # Create prompt
            full_prompt = boxing_prompt_template.format(context=context, question=question)
            
            # Generate response
            try:
                if hasattr(model, '_call'):
                    response = model._call(full_prompt)
                elif hasattr(model, 'invoke'):
                    response = model.invoke(full_prompt)
                else:
                    response = str(model(full_prompt))
            except Exception as model_error:
                print(f"⚠️ Model error: {model_error}")
                response = f"Basé sur le contexte fourni:\n{context[:500]}...\n\nJe ne peux pas générer une réponse complète, mais voici les informations trouvées."
            
            return {
                'result': response,
                'source_documents': docs
            }
        except Exception as rag_error:
            print(f"❌ RAG error: {rag_error}")
            return {
                'result': "Désolé, je ne peux pas traiter cette question pour le moment.",
                'source_documents': []
            }
    
    qa_chain = manual_rag
    print("✅ Manual RAG implementation ready!")

✅ Model validation successful: Test prompt. Use the 'nose' button to enable the p...
✅ RAG system ready for boxing questions!


In [30]:
# TEST
def test_rag_system(question: str):
    """Test the RAG system with a sample question"""
    try:
        result = qa_chain(question)
        print(f"Question: {question}")
        print(f"Réponse: {result['result'][:200]}...")  # Print first 200 chars
        print(f"Source documents: {[doc.metadata for doc in result['source_documents']]}")
    except Exception as e:
        print(f"❌ Error during RAG test: {e}")
        

# Example test questions
test_questions = [
    "Qui est le meilleur boxer ?"
]

for question in test_questions:
    test_rag_system(question)

Token indices sequence length is longer than the specified maximum sequence length for this model (1722 > 1024). Running this sequence through the model will result in indexing errors


❌ Error during RAG test: index out of range in self
